# M1 — Détection de fraude (Isolation Forest)

**Orange Money — Système ML | Koceila SALEM**

Modèle non supervisé sur 32 jours. Utilise l'architecture `src/` :
- `src.config` : colonnes, seuils, params
- `src.data_loader` : chargement Parquet
- `src.features` : 4 blocs A/B/C/D
- `src.utils` : scoring 0-100, niveaux

Pas de code dupliqué — tout vient des modules.

## 0. Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, json, time, warnings
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

from src import config as cfg
from src.data_loader import load_parquet, load_columns, cast_numeric
from src.features import bloc_a_transaction, bloc_b_temporel, bloc_c_comportemental, bloc_d_contextuel
from src import utils

MODEL_DIR  = cfg.MODELS_DIR / 'M1_fraude'
OUTPUT_DIR = cfg.OUTPUTS_DIR / 'M1_fraude'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Modules src/ chargés')
print(f'Racine : {ROOT}')

Modules src/ chargés
Racine : c:\Users\RQKB6834\OneDrive - orange.com\Bureau\Alternance\ML


## 1. Chargement des données (Parquet)

In [3]:
cols = load_columns('M1')
df = load_parquet(columns=cols)
df = cast_numeric(df)

Chargé en 68.9s
Dimensions : 25,456,467 lignes × 19 colonnes
RAM        : 17.45 Go
Plage      : 2025-08-29 10:35:20 → 2025-09-30 23:59:59
Jours      : 32


## 2. Tri chronologique (requis pour le Bloc C glissant)

In [4]:
debut = time.time()
df = df.sort_values([cfg.COL_SENDER_ID, cfg.COL_DATE]).reset_index(drop=True)
print(f'Trié en {time.time()-debut:.1f}s')
print(f'Comptes uniques : {df[cfg.COL_SENDER_ID].nunique():,}')

Trié en 103.8s
Comptes uniques : 1,161,043


## 3. Feature Engineering — 4 blocs (depuis src/features)

In [5]:
t = time.time()
df = bloc_a_transaction.build(df)
print(f'Bloc A OK | {time.time()-t:.0f}s')
df = bloc_b_temporel.build(df)
print(f'Bloc B OK | {time.time()-t:.0f}s')
df = bloc_c_comportemental.build(df, verbose=True)
df = bloc_d_contextuel.build(df)
print(f'Bloc D OK | {time.time()-t:.0f}s')

FEATURE_COLS = [c for c in df.columns if c.startswith('f_')]
print(f'\nTOTAL : {len(FEATURE_COLS)} features')

Bloc A OK | 9s
Bloc B OK | 15s
  vélocité 7j OK | 183s
  profil compte OK | 1487s
  762 ligne(s) __UNKNOWN__ neutralisée(s)
✅ Bloc C complet | 2077s
Bloc D OK | 2368s

TOTAL : 49 features


## 4. Préparation & scaling

In [6]:
X_df = df[FEATURE_COLS].copy()

# NaN -> médiane
nan_cols = X_df.isna().sum()
nan_cols = nan_cols[nan_cols > 0]
if len(nan_cols) > 0:
    print(f'NaN imputés : {list(nan_cols.index)}')
    X_df = X_df.fillna(X_df.median())

# Colonnes constantes -> supprimées
const = [c for c in FEATURE_COLS if X_df[c].std() == 0]
if const:
    print(f'Constantes supprimées : {const}')
    FEATURE_COLS = [c for c in FEATURE_COLS if c not in const]
    X_df = X_df[FEATURE_COLS]

print(f'Dataset final : {X_df.shape[0]:,} x {X_df.shape[1]}')
scaler = RobustScaler()
X = scaler.fit_transform(X_df)
print('RobustScaler appliqué')

Dataset final : 25,456,467 x 49
RobustScaler appliqué


## 5. Entraînement Isolation Forest

In [7]:
print('Entraînement...')
t = time.time()
iso = IsolationForest(**cfg.IFOREST_PARAMS)
iso.fit(X)
print(f'Entraîné en {time.time()-t:.0f}s sur {len(X):,} transactions')

scores = iso.score_samples(X)
preds  = iso.predict(X)
print(f'Anomalies : {(preds==-1).sum():,} ({(preds==-1).mean()*100:.2f}%)')

Entraînement...
Entraîné en 392s sur 25,456,467 transactions
Anomalies : 254,564 (1.00%)


## 6. Score 0-100 + niveaux (depuis src/utils)

In [ ]:
df['RISK_SCORE'], score_meta = utils.score_to_risk(scores, method='percentile')
df['IS_ANOMALY'] = (preds == -1).astype(int)
df = utils.apply_risk_levels(df)

print('Distribution RISK_SCORE :')
print(df['RISK_SCORE'].describe(percentiles=[.5,.9,.95,.99]).round(2))
print('\nNiveaux :')
print(df['RISK_LEVEL'].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(12,4))
ax.hist(df['RISK_SCORE'], bins=100, color='steelblue', edgecolor='white', alpha=0.8)
for s, c in [(cfg.SEUILS['operationnel'],'orange'), (cfg.SEUILS['eleve'],'red'), (cfg.SEUILS['critique'],'darkred')]:
    ax.axvline(s, color=c, linestyle='--')
ax.set_title('Distribution RISK_SCORE — M1 (32 jours)', fontweight='bold')
ax.set_xlabel('Score'); ax.set_ylabel('Transactions')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'score_dist.png', dpi=150); plt.show()

Distribution RISK_SCORE :
count    25456467.00
mean           50.00
std            28.87
min             0.00
50%            50.00
90%            90.00
95%            95.00
99%            99.00
max           100.00
Name: RISK_SCORE, dtype: float64

Niveaux :
RISK_LEVEL
Normal      17819526
Modéré       3818470
Élevé        2545647
Critique     1272824
Name: count, dtype: int64


## 7. Analyse temporelle des alertes (32 jours)

In [1]:
seuil = cfg.SEUILS['operationnel']
alertes = df[df['RISK_SCORE'] >= seuil]
print(f'Alertes : {len(alertes):,} ({len(alertes)/len(df)*100:.2f}%)')

df['_date'] = df[cfg.COL_DATE].dt.date
alertes_jour = df[df['RISK_SCORE'] >= seuil].groupby('_date').size()

fig, axes = plt.subplots(1, 2, figsize=(16,4))
alertes_jour.plot(ax=axes[0], color='tomato', marker='o', markersize=3)
axes[0].set_title('Alertes par jour (32j)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

df.groupby(df[cfg.COL_DATE].dt.hour)['RISK_SCORE'].mean().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Score moyen par heure', fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'alertes_temporel.png', dpi=150); plt.show()

NameError: name 'cfg' is not defined

## 8. Top comptes suspects

In [ ]:
top = (alertes.groupby(cfg.COL_SENDER_ID)
       .agg(nb=('RISK_SCORE','count'), score_max=('RISK_SCORE','max'),
            montant=(cfg.COL_MONTANT,'sum'), velocite=('f_velocite_7j','max'))
       .sort_values('nb', ascending=False).head(15))
print('Top 15 comptes suspects :')
print(top)

## 9. Importance des features

In [ ]:
print('Permutation importance (échantillon 50k)...')
idx = np.random.RandomState(42).choice(len(X), min(50_000,len(X)), replace=False)
Xs = X[idx]; ref = iso.score_samples(Xs)
imp = []
for j, f in enumerate(FEATURE_COLS):
    Xp = Xs.copy(); np.random.shuffle(Xp[:,j])
    imp.append({'feature':f, 'importance':np.abs(ref-iso.score_samples(Xp)).mean()})
df_imp = pd.DataFrame(imp).sort_values('importance', ascending=False)

top_n = min(25, len(df_imp))
fig, ax = plt.subplots(figsize=(10, top_n*0.38+1))
ax.barh(df_imp['feature'].head(top_n)[::-1], df_imp['importance'].head(top_n)[::-1], color='#1565C0')
ax.set_title('Top features — M1 Isolation Forest', fontweight='bold')
plt.tight_layout(); plt.savefig(OUTPUT_DIR/'feature_importance.png', dpi=150); plt.show()
print(df_imp.head(15).to_string(index=False))

## 10. Export (models/ + outputs/)

In [ ]:
# Modèle + scaler -> models/
joblib.dump(iso, MODEL_DIR/'iforest.pkl')
joblib.dump(scaler, MODEL_DIR/'scaler.pkl')

# Params -> models/
params = {
    'FEATURE_COLS': FEATURE_COLS, 'n_features': len(FEATURE_COLS),
    'SEUILS': cfg.SEUILS, 'score_calibration': score_meta,
    **cfg.IFOREST_PARAMS,
    'n_jours': int(df[cfg.COL_DATE].dt.date.nunique()),
    'n_transactions': int(len(df)),
}
with open(MODEL_DIR/'params.json','w') as f:
    json.dump(params, f, indent=2, default=str)

# Dataset scoré + alertes -> outputs/
id_cols = [cfg.COL_TRANSFER_ID, cfg.COL_DATE, cfg.COL_SENDER_ID, cfg.COL_RECVR_ID,
           cfg.COL_MONTANT, cfg.COL_SERVICE, cfg.COL_STATUT, cfg.COL_VILLE]
id_cols = [c for c in id_cols if c in df.columns]
export_cols = list(dict.fromkeys(id_cols + ['RISK_SCORE','RISK_LEVEL','IS_ANOMALY'] + FEATURE_COLS))

df[export_cols].to_parquet(OUTPUT_DIR/'scored.parquet', index=False)
df[df['RISK_SCORE']>=seuil][export_cols].sort_values('RISK_SCORE', ascending=False)\
  .to_csv(OUTPUT_DIR/'alertes.csv', index=False, encoding='utf-8-sig')

print('Exports créés :')
print(f'  models/M1_fraude/  : iforest.pkl, scaler.pkl, params.json')
print(f'  outputs/M1_fraude/ : scored.parquet, alertes.csv')
print(f'\n=== RÉSUMÉ M1 ===')
print(f'Transactions : {len(df):,} sur {params["n_jours"]} jours')
print(f'Features     : {len(FEATURE_COLS)}')
print(f'Alertes      : {(df["RISK_SCORE"]>=seuil).sum():,}')
print(f'Critiques    : {(df["RISK_SCORE"]>=cfg.SEUILS["critique"]).sum():,}')